In [ ]:
import time
global_start_time = time.time()
global_end_time = global_start_time + 12 * 3600 - 600         # inherited deadline; sampling stops 60 s before it
ttft_end_time = global_start_time + 9 * 3600                 # TTFT stage cap so sampling always keeps a share of the 12 h

In [ ]:
!python --version

In [ ]:
!nvidia-smi

In [ ]:
import torch
torch.__version__


In [ ]:
torch.version.cuda

In [ ]:
import transformers
transformers.__version__

In [ ]:
!pip install --no-deps --no-build-isolation /kaggle/input/notebooks/cpmpml/pip-dowload-flash/flash_attn-2.8.3+cu128torch2.11-cp313-cp313-linux_x86_64.whl

In [ ]:
import flash_attn
flash_attn.__version__

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
# 27B TTFT config for ttft_split_r1013.py (only the fields it reads). TTFT trains on FP8 block-128 of the GPTQ mixed weights (filled at load
# from the packed weights); sglang serves the same GPTQ mixed model. Both read the local copy that ttft27b_split.py makes in /kaggle/tmp/gptq. ttft27b_split.py swaps in the test challenges on the hidden rerun.
import json
GPTQ_27B = "/kaggle/tmp/gptq"   # copied from the attached dataset darraghdog/qwen38-27b-src11plus16abcd-step188222-gptq-mixed
TTFT_CFG = {"challenges": "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json",
    "models": {"27b": {"model": GPTQ_27B, "rank": 16, "alpha": 16, "modules_to_not_convert": ["lm_head", "embed_tokens", "in_proj_a", "in_proj_b", "conv1d"],
                        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}},
    "training": {"learning_rate": 5e-05},
    "arms": [{"id": "t16_top32_nohead", "steps": 48, "schedule": {"type": "wsd", "warmup_steps": 6, "stable_steps": 30, "decay_steps": 12, "min_lr_ratio": 0.0},
              "checkpoint_rules": [{"max_training_tokens": 600, "fraction": 0.0}, {"max_training_tokens": 99999, "fraction": 1.0}],
              "note": "TTFT48 WSD 6/30/12, LoRA r16 alpha 16 (PEFT init, no saved init) on q/k/v/o/gate/up/down of layers 32-63 only (no embed/lm_head LoRA, for sglang serving)"}]}
json.dump(TTFT_CFG, open("/kaggle/working/ttft27b_config.json", "w"), indent=1)

In [ ]:
# sglang 0.5.17 offline from the attached wheelhouse kernel, installed into its own dir (no venv) and used only via PYTHONPATH by sgl_stage.py
!pip install --no-index --no-deps --target /kaggle/tmp/sgl_site $(ls /kaggle/input/pip-dowload-flash-sglang/wheels/*.whl | grep -vE '/(torch|torchvision|torchaudio)-') 2>&1 | tail -2
p = "/kaggle/tmp/sgl_site/sglang/srt/models/qwen3_5_text.py"; s = open(p).read()
if "def get_hidden_dim" not in s:
    s = s.replace("    supported_lora_modules = qwen3_5.Qwen3_5ForCausalLM.supported_lora_modules\n",
                  "    supported_lora_modules = qwen3_5.Qwen3_5ForCausalLM.supported_lora_modules\n\n    def get_hidden_dim(self, module_name, layer_idx):  # arc2 patch: delegate to the inner model (honours attn_output_gate)\n        return self.model.get_hidden_dim(module_name, layer_idx)\n", 1)
    open(p, "w").write(s); print("PATCHED qwen3_5_text.get_hidden_dim")
else:
    print("get_hidden_dim already present")



In [ ]:
%%writefile arc_loader.py
# Adapted from the ARChitects' ARC Prize 2024 code (https://github.com/da-fr/arc-prize-2024, Apache-2.0,
# Copyright 2024 Daniel Franzen and Jan Disselhoff) via NVARC. Modified by NVARC and the NVBANANA team; see THIRD_PARTY_NOTICES.md.
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
# Adapted from the ARChitects' ARC Prize 2024 code (https://github.com/da-fr/arc-prize-2024, Apache-2.0,
# Copyright 2024 Daniel Franzen and Jan Disselhoff) via NVARC. getter_kgmon/score_kgmon come from NVARC.
# Modified by the NVBANANA team; see THIRD_PARTY_NOTICES.md.
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile ttft_split_r1013.py
"""Split-prefix TTFT: one GPU serves the frozen bottom half (layers 0-31) to several top-half trainers.

bottom (GPU 0): 32-layer truncated model, LoRA attached with B=0 (same kernels as production), forward under no_grad,
                returns the layer-31 output for each requested sample, earliest step first.
top (GPU 1..N): 32-layer model built from layers 32-63, production initial LoRA, production TTFT loop; the embedding
                output is replaced by the bottom's hidden state, so layers 32-63 see exactly the production input.
Adapters are saved with the original layer indices (+split). Eval puzzles are used for timing only.
"""
import argparse, contextlib, heapq, json, math, os, queue, shutil, sys, time, hashlib
from pathlib import Path
import torch
import torch.multiprocessing as mp


def setup(gpu, run, tag):
    os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu)
    for k, sub in (('TRITON_CACHE_DIR', 'triton'), ('TORCHINDUCTOR_CACHE_DIR', 'inductor'), ('UNSLOTH_COMPILE_LOCATION', 'unsloth')):
        os.environ[k] = str(Path(run)/f'{sub}_{tag}')


def unpack_dequant(packed, scale, shape, device='cuda:0'):
    """compressed-tensors pack-quantized, symmetric int, group strategy: int32 words hold 32/bits values, lowest bits first,
    stored with a +2^(bits-1) offset; weight = x_q * scale[:, col // group] in the scale dtype (bf16)."""
    rows, cols = int(shape[0]), int(shape[1])
    p = packed.to(device); bits = 32 * p.shape[1] // cols; pf = 32 // bits; mask = (1 << bits) - 1
    assert 32 % bits == 0 and p.shape[1] * pf >= cols and p.shape[0] == rows, (p.shape, shape, bits)
    q = torch.empty(rows, p.shape[1] * pf, dtype=torch.int32, device=device)
    for i in range(pf):
        q[:, i::pf] = (p >> (bits * i)) & mask
    q = q[:, :cols] - (1 << (bits - 1))
    s = scale.to(device); group = cols // s.shape[1]
    return q.to(s.dtype) * s.repeat_interleave(group, dim=1), bits


@torch.no_grad()
def fill_from_gptq(model, path):
    """a half built as symlinks into a GPTQ-mixed export (gptq_source.json; config carries the TTFT FP8 quantization_config, so
    it loads as pre-quantized FP8 and the absent linears are created FP8-sized) has no `weight` for the packed linears. Fill every
    such FP8Linear: unpack -> bf16 -> transformers' Fp8Quantize (the load-time FP8 op) with the model's own quantizer config, so the result
    equals converting that bf16 weight at load. Layer names are mapped back by the marker's offset (top half: i -> i + 32), for the
    linears and for the top half's plain per-layer tensors."""
    from safetensors import safe_open
    from transformers.integrations.finegrained_fp8 import Fp8Quantize
    path = Path(path); src = json.loads((path/'gptq_source.json').read_text())
    wm = json.loads((path/'model.safetensors.index.json').read_text())['weight_map']
    off, pre = src['layer_offset'], src['layer_prefix']
    op = Fp8Quantize(model.hf_quantizer); handles = {}
    def get(k):
        if wm[k] not in handles: handles[wm[k]] = safe_open(str(path/wm[k]), 'pt', device='cpu')
        return handles[wm[k]].get_tensor(k)
    t0 = time.time(); n = {4: 0, 8: 0}
    for name, m in model.named_modules():
        if type(m).__name__ != 'FP8Linear':
            continue
        ck = name
        if off and f'{pre}.layers.' in name:
            li = int(name.split('.layers.')[1].split('.')[0]); ck = name.replace(f'{pre}.layers.{li}.', f'{pre}.layers.{li + off}.', 1)
        # the half loads as pre-quantized FP8 (builder config), so a bf16 `weight` here would be taken as FP8 bytes: packed only
        assert f'{ck}.weight_packed' in wm and f'{ck}.weight' not in wm, ck
        w, bits = unpack_dequant(get(f'{ck}.weight_packed'), get(f'{ck}.weight_scale'), get(f'{ck}.weight_shape').tolist())
        out = op.convert({'x.weight': [w]})   # transformers 5.5 (Kaggle) and 5.8 (training cluster): same op, same result
        dev = m.weight_scale_inv.device if m.weight_scale_inv.device.type != 'meta' else torch.device('cuda:0')
        m.weight = torch.nn.Parameter(out['x.weight'].to(dev), requires_grad=False)
        m.weight_scale_inv = torch.nn.Parameter(out['x.weight_scale_inv'].to(dev), requires_grad=False)
        n[bits] += 1
    copied = 0
    if off:   # top half: the plain per-layer tensors (norms, GDN dt_bias/A_log/conv1d/in_proj_a/b) from layers.<i + off>; unsloth's
              # quantized-model load drops the key_mapping, so the load put layers.<i> of the bottom half there
        for name, t in list(model.named_parameters()) + list(model.named_buffers()):
            if f'{pre}.layers.' not in name:
                continue
            li = int(name.split('.layers.')[1].split('.')[0]); ck = name.replace(f'{pre}.layers.{li}.', f'{pre}.layers.{li + off}.', 1)
            if ck in wm:
                t.data.copy_(get(ck).to(t.device)); copied += 1
    print(f'GPTQ_FILL int4 {n[4]} int8 {n[8]} plain_copied {copied} {time.time() - t0:.1f}s', flush=True)
    return n


class _skip_fp8_init:
    """transformers 5.5 (Kaggle) random-initializes the absent, packed-only FP8 linears of a GPTQ half at load; CUDA has no
    FP8 kernels for those init ops, and fill_from_gptq overwrites the weights anyway. Inside this context FP8 tensors skip them."""
    def __enter__(self):
        self.orig = {n: getattr(torch.Tensor, n) for n in ('normal_', 'uniform_', 'erfinv_', 'mul_', 'add_', 'clamp_', 'fill_', 'zero_')}   # init ops (incl. trunc_normal_)
        def wrap(f):
            def g(t, *a, **k):
                return t if t.dtype in (torch.float8_e4m3fn, torch.float8_e5m2) else f(t, *a, **k)
            return g
        for n, f in self.orig.items(): setattr(torch.Tensor, n, wrap(f))
    def __exit__(self, *exc):
        for n, f in self.orig.items(): setattr(torch.Tensor, n, f)


def load_half(path, spec, max_seq=8192):
    from unsloth import FastLanguageModel
    from transformers import FineGrainedFP8Config, set_seed
    set_seed(42)
    q = FineGrainedFP8Config(weight_block_size=(128, 128), modules_to_not_convert=spec['modules_to_not_convert'])
    gptq = (Path(path)/'gptq_source.json').exists()   # half symlinked into the GPTQ mixed model: FP8 linears from the packed weights
    with (_skip_fp8_init() if gptq else contextlib.nullcontext()):
        model, _ = FastLanguageModel.from_pretrained(model_name=path, max_seq_length=max_seq, dtype=torch.bfloat16, load_in_4bit=False,
            load_in_8bit=False, load_in_16bit=False, load_in_fp8=False, full_finetuning=False, fast_inference=False,
            local_files_only=True, device_map={'': 0}, unsloth_tiled_mlp=False, quantization_config=q)
    if gptq:
        fill_from_gptq(model, path)
    if os.environ.get('ARC2_SPLIT_HOOKS') == '1':
        # Production loads 64 layers over 2 GPUs, so accelerate wraps every submodule in an AlignDevicesHook before LoRA is added.
        # The hooks change how unsloth's compiled MLP is split into graphs (and so its rounding); attach the same hooks here.
        from accelerate import dispatch_model
        n = model.config.num_hidden_layers
        mapping = {'model.embed_tokens': 0, 'model.rotary_emb': 0, 'model.norm': 0, 'lm_head': 0, **{f'model.layers.{i}': 0 for i in range(n)}}
        model = dispatch_model(model, device_map=mapping, force_hooks=True)
    model = FastLanguageModel.get_peft_model(model, r=spec['rank'], lora_alpha=spec.get('alpha', 32), lora_dropout=0, bias='none',
        target_modules=spec['target_modules'], use_gradient_checkpointing=True, random_state=42, use_rslora=True, loftq_config=None)
    return model


class _Stop(Exception):
    pass


def bottom_main(a, req_q, res_qs, ctl_q):
    setup(a.gpus[0], a.output, 'bottom')
    from unsloth import FastLanguageModel
    cfg = json.loads(Path(a.config).read_text()); spec = cfg['models']['27b']
    t0 = time.perf_counter(); model = load_half(a.bottom_model, spec, a.max_seq)
    for p in model.parameters():
        p.requires_grad_(False)
    model = FastLanguageModel.for_training(model, use_gradient_checkpointing=False)
    model.config.use_cache = False
    layers = model.base_model.model.model.layers
    for layer in layers:
        layer.gradient_checkpointing = False
    last = layers[len(layers)-1]; box = {}
    orig = last.forward
    def stop_forward(*x, **k):
        box['out'] = orig(*x, **k); raise _Stop
    last.forward = stop_forward
    ctl_q.put(('ready', 'bottom', time.perf_counter()-t0)); print('BOTTOM loaded', round(time.perf_counter()-t0, 1), 's', flush=True)
    heap, seq, busy, served, stop = [], 0, 0.0, 0, False
    while True:
        try:
            block = not heap and not stop
            while True:
                item = req_q.get(block=block, timeout=None if block else 0)
                block = False
                if item is None:
                    stop = True
                else:
                    heapq.heappush(heap, (item[2], seq, item)); seq += 1
        except queue.Empty:
            pass
        if not heap:
            if stop:
                break
            continue
        _, _, (w, key, step, batch) = heapq.heappop(heap)
        s = time.perf_counter()
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            try:
                model(**{k: v.to('cuda:0') for k, v in batch.items()}, use_cache=False)
            except _Stop:
                pass
        out = box.pop('out'); out = out if torch.is_tensor(out) else out[0]
        hidden = out.detach().to('cpu')
        busy += time.perf_counter()-s; served += 1
        res_qs[w].put((key, step, hidden))
    ctl_q.put(('bottom_done', served, busy)); print('BOTTOM done; served', served, 'busy', round(busy, 1), 's', flush=True)


def top_main(w, a, task_q, req_q, res_q, ctl_q):
    setup(a.gpus[w+1], a.output, f'top{w}')
    from unsloth import FastLanguageModel
    from safetensors.torch import save_file
    from profile_qwen35_unsloth_ttft_l4 import load_qwen2_fast_tokenizer, QwenCompletionCollator, build_train_dataset, ArcDataset, QwenFormatter
    import l4_host_staged_dispatch_20261001 as transport
    from l4_e2e_20261001.worker_v5 import make_scheduler, apply_checkpointing
    from l4_e2e_20261001.optimizations_v4 import checkpoint_fraction_for
    cfg = json.loads(Path(a.config).read_text()); spec = cfg['models']['27b']; arm = next(x for x in cfg['arms'] if x['id'] == a.arm)
    challenges = json.loads(Path(cfg['challenges']).read_text())
    tokenizer = load_qwen2_fast_tokenizer(spec['model'], True)
    t0 = time.perf_counter(); model = load_half(a.top_model, spec, a.max_seq)
    lora = {n: p for n, p in model.named_parameters() if 'lora_' in n}
    if a.init_lora != 'none':   # 'none' keeps the PEFT init (seed 42, B=0); the saved init is rank 32 only
        init = torch.load(a.init_lora, map_location='cpu', weights_only=True)
        assert lora.keys() == init.keys(), (len(lora), len(init), sorted(set(lora) ^ set(init))[:4])
        with torch.no_grad():
            for n, p in lora.items():
                p.copy_(init[n])
    parameters = {n: p for n, p in model.named_parameters() if p.requires_grad}
    assert set(parameters) == set(lora) and sum(p.numel() for p in parameters.values()) == 79691776 * spec['rank'] // 32, sum(p.numel() for p in parameters.values())
    baseline = {n: p.detach().cpu().clone() for n, p in parameters.items()}
    embed = model.get_input_embeddings(); inject = {}
    embed.forward = lambda input_ids, *x, **k: inject['h']
    formatter = QwenFormatter(tokenizer); collator = QwenCompletionCollator(tokenizer)
    count = arm.get('steps', 16); schedule = arm.get('schedule', {'type': 'cosine'})
    peft_cfg = model.peft_config['default']; r = peft_cfg.r; alpha = peft_cfg.lora_alpha
    ctl_q.put(('ready', f'top{w}', time.perf_counter()-t0)); print(f'TOP{w} loaded', round(time.perf_counter()-t0, 1), 's', f'lora rank {r} alpha {alpha} init {a.init_lora}', flush=True)

    def prepare(key):
        task = challenges[key]
        demos = {'train': task['train'], 'test': [{'input': [[0]]}]}
        dataset, stats = build_train_dataset(ArcDataset({key: demos}), key, formatter, a.max_seq, 0, 0)
        order = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(42)).tolist()[:count]
        batches = [collator([dataset[i]]) for i in order]
        for step, b in enumerate(batches, 1):
            req_q.put((w, key, step, {k: v for k, v in b.items()}))
        return {'key': key, 'stats': stats, 'batches': batches}

    report = {'worker': w, 'puzzles': {}}; got = {}
    nxt = task_q.get(); cur = prepare(nxt) if nxt is not None else None
    while cur is not None:
        key, stats, batches = cur['key'], cur['stats'], cur['batches']
        fraction = checkpoint_fraction_for(arm['checkpoint_rules'], stats['max_tokens']) if arm.get('checkpoint_rules') else 1.0
        model = FastLanguageModel.for_training(model, use_gradient_checkpointing=True)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True}); model.config.use_cache = False
        ck = apply_checkpointing(model, fraction)
        optimizer = torch.optim.AdamW(list(parameters.values()), lr=cfg['training']['learning_rate'], weight_decay=0, fused=True)
        scheduler, _ = make_scheduler(optimizer, schedule, count)
        rows, wait, upcoming = [], 0.0, None
        torch.cuda.synchronize(); tp = time.perf_counter()
        for step, b in enumerate(batches, 1):
            if step == max(1, count-3) and upcoming is None:
                k2 = task_q.get(); upcoming = prepare(k2) if k2 is not None else 'end'
            s = time.perf_counter()
            while (key, step) not in got:
                k, st, h = res_q.get(); got[(k, st)] = h
            wait += time.perf_counter()-s
            inject['h'] = got.pop((key, step)).to('cuda:0', non_blocking=True).requires_grad_(True)
            optimizer.zero_grad(set_to_none=True)
            batch = {k: v.to('cuda:0') for k, v in b.items()}
            with torch.autocast('cuda', dtype=torch.bfloat16):
                loss = model(**batch, use_cache=False).loss
            loss.backward(); transport.clip_grad_norm(list(parameters.values()), 1); optimizer.step(); scheduler.step()
            rows.append({'step': step, 'loss': loss.item()})
        optimizer.zero_grad(set_to_none=True); inject.clear()
        torch.cuda.synchronize(); ttft = time.perf_counter()-tp
        # ---- save adapter (PEFT layout, original layer indices) ----
        state, kept = {}, set()
        for n, p in model.named_parameters():
            if 'lora_' not in n:
                continue
            li = int(n.split('.layers.')[1].split('.')[0]); kept.add(li+a.split)
            name = n.replace(f'.layers.{li}.', f'.layers.{li+a.split}.', 1).replace('.default', '')
            t = p.detach().to(torch.bfloat16).cpu()
            if '.lora_B.' in name:
                t = (t.float()*alpha/math.sqrt(r)).to(torch.bfloat16)   # rsLoRA scale alpha/sqrt(r) folded into B (as production)
            state[name] = t.contiguous()
        adir = Path(a.adapter_dir)/key; adir.mkdir(parents=True, exist_ok=True)
        save_file(state, str(adir/'adapter_model.safetensors'))
        acfg = {'peft_type': 'LORA', 'base_model_name_or_path': spec['model'], 'r': r, 'lora_alpha': r, 'lora_dropout': 0.0, 'bias': 'none',
                'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                'layers_to_transform': sorted(kept), 'use_rslora': False, 'task_type': 'CAUSAL_LM', 'inference_mode': True}
        (adir/'adapter_config.json').write_text(json.dumps(acfg, indent=2))
        dst = Path(a.output)/'adapters'/key
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(adir, dst)
        report['puzzles'][key] = {'training_tokens': stats['max_tokens'], 'checkpointed': ck['checkpointed'], 'ttft_s': ttft, 'wait_s': wait,
                                  'loss': rows, 'sha256': hashlib.sha256((adir/'adapter_model.safetensors').read_bytes()).hexdigest()}
        print(f'TOP{w} PUZZLE', key, 'tok', stats['max_tokens'], 'ttft', round(ttft, 1), 's; waited', round(wait, 2), 's', flush=True)
        with torch.no_grad():
            for n, v in baseline.items():
                parameters[n].copy_(v); parameters[n].grad = None
        del optimizer, scheduler
        Path(a.output, f'report_top{w}.json').write_text(json.dumps(report, indent=1))
        if upcoming is None:
            k2 = task_q.get(); upcoming = prepare(k2) if k2 is not None else 'end'
        cur = None if upcoming == 'end' else upcoming
    ctl_q.put(('top_done', w, None)); print(f'TOP{w} DONE', flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', required=True); ap.add_argument('--arm', required=True); ap.add_argument('--keys', nargs='+', required=True)
    ap.add_argument('--bottom-model', required=True); ap.add_argument('--top-model', required=True); ap.add_argument('--init-lora', required=True)
    ap.add_argument('--gpus', nargs='+', type=int, default=[0, 1, 2, 3]); ap.add_argument('--split', type=int, default=32)
    ap.add_argument('--max-seq', type=int, default=8192)
    ap.add_argument('--output', required=True); ap.add_argument('--adapter-dir', default='/dev/shm/arc2_split_adapters')
    a = ap.parse_args(); Path(a.output).mkdir(parents=True, exist_ok=True); Path(a.adapter_dir).mkdir(parents=True, exist_ok=True)
    ctx = mp.get_context('spawn'); n_top = len(a.gpus)-1
    req_q, task_q, ctl_q = ctx.Queue(), ctx.Queue(), ctx.Queue(); res_qs = [ctx.Queue() for _ in range(n_top)]
    procs = [ctx.Process(target=bottom_main, args=(a, req_q, res_qs, ctl_q))]
    procs += [ctx.Process(target=top_main, args=(w, a, task_q, req_q, res_qs[w], ctl_q)) for w in range(n_top)]
    for p in procs:
        p.start()
    ready = 0
    while ready < len(procs):
        msg = ctl_q.get(); print('CTL', msg, flush=True); ready += msg[0] == 'ready'
    t0 = time.perf_counter()
    for k in a.keys:
        task_q.put(k)
    for _ in range(n_top):
        task_q.put(None)
    done = 0
    while done < n_top:
        msg = ctl_q.get(); print('CTL', msg, flush=True); done += msg[0] == 'top_done'
    wall = time.perf_counter()-t0
    req_q.put(None)
    msg = ctl_q.get(); print('CTL', msg, flush=True)
    for p in procs:
        p.join()
    print('SPLIT_DONE wall', round(wall, 1), 's for', len(a.keys), 'puzzles', flush=True)
    Path(a.output, 'summary.json').write_text(json.dumps({'wall_s': wall, 'keys': a.keys, 'bottom': msg}, indent=1))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile ttft27b_split.py
"""27B split-prefix TTFT on Kaggle 4xL4: GPU 0 runs the frozen bottom half (layers 0-31)
for three top-half trainers on GPUs 1-3 (layers 32-63, schedule and LoRA rank from the TTFT config, FP8 frozen base, MAX_SEQ 8192), via
ttft_split_r1013.py. The GPTQ mixed model and its half configs (dataset qwen38-27b-src11plus16abcd-step188222-gptq-mixed) are copied
to /kaggle/tmp; both halves are configs + symlinks to the local shards, with the FP8 linears filled from the packed GPTQ weights at load.
PEFT adapters end in /kaggle/tmp/adapters/<key>."""
import os, sys, json, time, glob, shutil, signal, subprocess, argparse

def build_bottom(src, dst, split=32):   # same as prep_split_checkpoints.py (bottom): config with 32 layers + filtered index over symlinked shards
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        if f.endswith((".json", ".j2", ".jinja", ".txt", ".model")) and not f.startswith("model.safetensors"):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
    c = json.load(open(f"{src}/config.json")); t = c.get("text_config", c)
    assert t["num_hidden_layers"] == 2 * split and t["layer_types"][:split] == t["layer_types"][split:]
    t["num_hidden_layers"] = split; t["layer_types"] = t["layer_types"][:split]
    if "mtp_num_hidden_layers" in t: t["mtp_num_hidden_layers"] = 0
    json.dump(c, open(f"{dst}/config.json", "w"), indent=2)
    index = json.load(open(f"{src}/model.safetensors.index.json"))["weight_map"]
    layer = lambda k: int(k.split(".layers.")[1].split(".")[0]) if ".layers." in k else None
    wm = {k: v for k, v in index.items() if (layer(k) is None and not k.startswith("mtp")) or (layer(k) is not None and layer(k) < split)}
    for shard in sorted(set(wm.values())):
        if not os.path.exists(f"{dst}/{shard}"): os.symlink(f"{src}/{shard}", f"{dst}/{shard}")
    json.dump({"metadata": {}, "weight_map": wm}, open(f"{dst}/model.safetensors.index.json", "w"), indent=1)
    print(f"BOTTOM_BUILT {len(wm)} tensors over {len(set(wm.values()))} shards -> {dst}", flush=True)

def copy_gptq(src, dst="/kaggle/tmp/gptq"):   # GPTQ mixed model + split-TTFT half configs (bottom/, top/) from the dataset to local disk
    import zipfile
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time(); os.makedirs(dst, exist_ok=True)
    files = sorted(f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f)) and not f.endswith(".zip"))
    CHUNK = 256 << 20   # 256 MB byte ranges, 16 readers in parallel across all files (shards are 3-4 GB)
    jobs = []
    for f in files:
        n = os.path.getsize(os.path.join(src, f))
        with open(os.path.join(dst, f), "wb") as fh: fh.truncate(n)
        jobs += [(f, off, min(CHUNK, n - off)) for off in range(0, n, CHUNK)]
    def copy_range(job):
        f, off, n = job
        with open(os.path.join(src, f), "rb") as fi, open(os.path.join(dst, f), "r+b") as fo:
            fi.seek(off); fo.seek(off)
            while n > 0:
                b = fi.read(min(n, 64 << 20)); assert b, (f, off, n); fo.write(b); n -= len(b)
    with ThreadPoolExecutor(16) as ex: list(ex.map(copy_range, jobs))
    bad = [f for f in files if os.path.getsize(os.path.join(src, f)) != os.path.getsize(os.path.join(dst, f))]; assert not bad, bad
    shards = sorted(set(json.load(open(f"{dst}/model.safetensors.index.json"))["weight_map"].values()))
    halves = []
    for h in ("bottom", "top"):   # half = config (32 layers, FP8 quantization_config) + filtered index + gptq_source.json + symlinks to the local shards
        d = f"/kaggle/tmp/split32_{h}"; os.makedirs(d, exist_ok=True)
        if os.path.isdir(f"{src}/{h}"): [shutil.copyfile(f"{src}/{h}/{f}", f"{d}/{f}") for f in os.listdir(f"{src}/{h}")]
        else: zipfile.ZipFile(f"{src}/{h}.zip").extractall(d)   # the dataset upload may keep the folder as a zip
        if os.path.isdir(f"{d}/{h}"):
            for f in os.listdir(f"{d}/{h}"): shutil.move(f"{d}/{h}/{f}", f"{d}/{f}")
            os.rmdir(f"{d}/{h}")
        assert all(os.path.exists(f"{d}/{f}") for f in ("config.json", "gptq_source.json", "model.safetensors.index.json")), (d, os.listdir(d))
        for s in shards: os.symlink(f"{dst}/{s}", f"{d}/{s}")
        halves.append(d)
    gb = sum(os.path.getsize(f"{dst}/{s}") for s in shards) / 1e9
    print(f"GPTQ_COPIED {src} -> {dst}: {len(files)} files, {gb:.1f} GB in {time.time() - t0:.0f}s; halves {halves}", flush=True)
    return dst, halves[0], halves[1]

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--end-time", type=float, default=0.0); ap.add_argument("--arm", default="t16_top32_nohead"); a = ap.parse_args()
    cfg_path = "/kaggle/working/ttft27b_config.json"; cfg = json.load(open(cfg_path))
    RT = glob.glob("/kaggle/input/**/runtime-fp8", recursive=True)[0]; DS = os.path.dirname(RT); SCRIPTS = DS + "/scripts"; DFS = DS + "/dfs_kernel_source"
    rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/" + ("arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json")
    if rerun: cfg["challenges"] = test_path; json.dump(cfg, open(cfg_path, "w"), indent=1)
    tasks = json.load(open(test_path)); keys = sorted(tasks.keys())
    smoke_keys = os.getenv("SMOKE_KEYS", "").split()
    if smoke_keys: keys = [k for k in keys if k in smoke_keys]; print(f"SMOKE run: {len(keys)} puzzles {keys}", flush=True)
    # Longest first: grid cells of all demo pairs + test inputs as the size proxy.
    size = lambda k: sum(len(g) * len(g[0]) for p in tasks[k]["train"] for g in (p["input"], p["output"])) + sum(len(p["input"]) * len(p["input"][0]) for p in tasks[k]["test"])
    keys = sorted(keys, key=lambda k: (-size(k), k))
    TTFT_MAX_SEQ = 8192
    SRC = os.path.dirname(next(p for p in glob.glob("/kaggle/input/**/PROVENANCE.json", recursive=True) if "src11plus16abcd-step188222-gptq-mixed" in p))
    GPTQ, BOTTOM, TOP = copy_gptq(SRC); assert GPTQ == cfg["models"]["27b"]["model"], cfg["models"]["27b"]["model"]
    assert cfg["models"]["27b"]["rank"] != 32; INIT = "none"   # a saved initial LoRA exists only for rank 32 and is not attached; adapters use PEFT init
    # Page-cache prefetch (lossless): the local GPTQ shards (both TTFT halves and the sglang serving model read them).
    import threading
    from concurrent.futures import ThreadPoolExecutor
    def prefetch(path, workers=16):
        t0 = time.time(); files = [os.path.join(r, f) for r, _, fs in os.walk(path) for f in fs if f.endswith(".safetensors")]
        def read(fp):
            with open(fp, "rb") as fh:
                while fh.read(1 << 30): pass
        with ThreadPoolExecutor(max_workers=workers) as ex: list(ex.map(read, files))
        print(f"PREFETCH {path.rstrip('/').split('/')[-1]} {sum(os.path.getsize(f) for f in files)/1e9:.1f} GB in {time.time()-t0:.0f}s", flush=True)
    threading.Thread(target=lambda: prefetch(GPTQ), daemon=True).start()
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"; env["PYTHONPATH"] = f"{RT}:{SCRIPTS}:{DFS}"
    env.update({"UNSLOTH_DISABLE_STATISTICS": "1", "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas", "OMP_NUM_THREADS": "4", "HF_HUB_OFFLINE": "1",
                "TRANSFORMERS_OFFLINE": "1", "PYTHONUNBUFFERED": "1", "PYTHONHASHSEED": "0", "TOKENIZERS_PARALLELISM": "false"})
    out = "/kaggle/tmp/ttft27b_split"; os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "ttft_split_r1013.py", "--config", cfg_path, "--arm", a.arm, "--max-seq", str(TTFT_MAX_SEQ),
           "--bottom-model", BOTTOM, "--top-model", TOP, "--init-lora", INIT, "--output", out, "--adapter-dir", "/dev/shm/arc2_split_adapters", "--keys", *keys]
    log_path = "/kaggle/working/ttft27b_split.log"; log = open(log_path, "w")
    p = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd="/kaggle/working", start_new_session=True)
    print(f"SPLIT started: 1 bottom + 3 top GPUs, {len(keys)} puzzles, longest first, lora rank {cfg['models']['27b']['rank']} alpha {cfg['models']['27b'].get('alpha', 32)}", flush=True)
    seen = 0
    while p.poll() is None:
        time.sleep(20)
        lines = open(log_path, errors="ignore").read().splitlines()
        for l in lines[seen:]:
            if l.startswith(("TOP", "BOTTOM", "CTL", "SPLIT_DONE", "Traceback")) or "Error" in l: print(l[:200], flush=True)
        seen = len(lines)
        if a.end_time and time.time() > a.end_time - 600:
            print("TIME_UP: stopping split TTFT", flush=True); os.killpg(p.pid, signal.SIGTERM); time.sleep(30)
            if p.poll() is None: os.killpg(p.pid, signal.SIGKILL)
            break
    print(f"SPLIT exit {p.returncode}; log tail:", flush=True); print("\n".join(open(log_path, errors="ignore").read().splitlines()[-8:]), flush=True)
    os.makedirs("/kaggle/tmp/adapters", exist_ok=True)
    for d in glob.glob(f"{out}/adapters/*"):
        if not os.path.exists(f"{d}/adapter_model.safetensors"): continue
        dst = f"/kaggle/tmp/adapters/{os.path.basename(d)}"
        if os.path.isdir(dst): shutil.rmtree(dst)
        shutil.move(d, dst)
    done = sorted(os.listdir("/kaggle/tmp/adapters"))
    print(f"TTFT_DONE adapters {len(done)} / {len(keys)}", flush=True)
    if len(done) < len(keys): print("TTFT_MISSING", sorted(set(keys) - set(done)), flush=True)

if __name__ == "__main__":
    main()


In [ ]:
import os
# Interactive commit: an 8-puzzle smoke on evaluation tasks. Hidden rerun: every task in arc-agi_test_challenges.json.
SMOKE_KEYS = "" if os.getenv("KAGGLE_IS_COMPETITION_RERUN") else "0934a4d8 135a2760 136b0064 13e47133 142ca369 16b78196 16de56c4 1818057f"
!SMOKE_KEYS="{SMOKE_KEYS}" python ttft27b_split.py --end-time {ttft_end_time}

In [ ]:
%%writefile sgl_stage.py
"""Sampling stage: two TP=2 sglang servers (GPUs 0,1 and 2,3) serving the GPTQ mixed (compressed-tensors) base with the per-puzzle
bf16 LoRA adapters saved by the TTFT stage; 16 augmented views x N samples per test input at T=1, majority vote by count via the
existing ArcDecoder (beam_score=0, score_aug=[0] so score_kgmon == count). Puzzle-level work stealing across the 2 servers."""
import os, sys, json, time, glob, bz2, pickle, queue, shutil, threading, subprocess, argparse, collections
from concurrent.futures import ThreadPoolExecutor
import requests
from arc_loader import ArcDataset, QwenFormatter

GPTQ_MODEL = "/kaggle/tmp/gptq"   # local copy of the GPTQ mixed dataset made by ttft27b_split.py
TOKENIZER = GPTQ_MODEL  # the 27B ships its own 17-token tokenizer
ADAPTER_DIR = "/kaggle/tmp/adapters"
OUT_DIR = "/kaggle/inference_outputs"
LOG_DIR = "/kaggle/tmp/sgl_logs"
EOS_ID = 15            # <|im_end|> in the 16-token vocabulary (ends every reply)
MAX_SEQ = 9216         # sampling cap (prompt cut 8285; server context 10240); TTFT trains at 8192
GPU_GROUPS = ["0,1", "2,3"]   # one TP=2 server per L4 pair
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


class LegacyQwenRoleTokenizer:
    """Role-marker-aware shim (same as profile_qwen35_unsloth_ttft_l4.load_qwen2_fast_tokenizer in the TTFT runtime): newer
    Transformers can load the old ARC tokenizer so that 'user'/'assistant' ids are dropped; restore [14,11,10]/[14,12,10]."""
    REPLACEMENTS = (("<|im_start|>assistant\n", [14, 12, 10]), ("<|im_start|>user\n", [14, 11, 10]), ("<|im_end|>", [15]))
    def __init__(self, base): self.base_tokenizer = base
    def __getattr__(self, name): return getattr(self.base_tokenizer, name)
    def encode(self, text, add_special_tokens=False, **kw):
        out, pos = [], 0
        while pos < len(text):
            nxt = None
            for marker, ids in self.REPLACEMENTS:
                i = text.find(marker, pos)
                if i >= 0 and (nxt is None or i < nxt[0]): nxt = (i, marker, ids)
            if nxt is None:
                out.extend(self.base_tokenizer.encode(text[pos:], add_special_tokens=False, **kw)); break
            i, marker, ids = nxt
            if i > pos: out.extend(self.base_tokenizer.encode(text[pos:i], add_special_tokens=False, **kw))
            out.extend(ids); pos = i + len(marker)
        return out
    def decode(self, *a, **kw): return self.base_tokenizer.decode(*a, **kw)


def load_qwen2_fast_tokenizer(model_path):
    from transformers import PreTrainedTokenizerFast
    try:
        from transformers.models.qwen2.tokenization_qwen2_fast import Qwen2TokenizerFast
        tok = Qwen2TokenizerFast.from_pretrained(model_path, local_files_only=True)
    except Exception:
        tok = PreTrainedTokenizerFast.from_pretrained(model_path, local_files_only=True)
    u = tok.encode("<|im_start|>user\n", add_special_tokens=False); a = tok.encode("<|im_start|>assistant\n", add_special_tokens=False)
    if u == a:
        tok = LegacyQwenRoleTokenizer(tok); u = tok.encode("<|im_start|>user\n"); a = tok.encode("<|im_start|>assistant\n")
    if u != [14, 11, 10] or a != [14, 12, 10]: raise RuntimeError(f"Unexpected role token ids: user={u}, assistant={a}")
    print("TOKENIZER role ids ok", flush=True)
    return tok


def sgl_env():
    site = "/kaggle/tmp/sgl_site"
    env = dict(os.environ)
    env["PYTHONPATH"] = site + ":" + env.get("PYTHONPATH", "")
    env["LD_LIBRARY_PATH"] = ":".join(glob.glob(site + "/nvidia/*/lib")) + ":" + env.get("LD_LIBRARY_PATH", "")
    env.setdefault("TRITON_PTXAS_PATH", "/usr/local/cuda/bin/ptxas")
    env["TRITON_CACHE_DIR"] = "/kaggle/tmp/triton_sgl"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["PYTHONUNBUFFERED"] = "1"
    # flashinfer JIT links with -lcuda; the Kaggle image has the driver's libcuda.so.1 but no libcuda.so (without it the build fails with
    # "ld: cannot find -lcuda"). Provide a stub dir with the symlink and put it on the compile-time library path.
    stub = "/kaggle/tmp/cudastub"
    if not os.path.exists(stub + "/libcuda.so"):
        os.makedirs(stub, exist_ok=True)
        found = subprocess.run("ls /usr/lib/x86_64-linux-gnu/libcuda.so.1 /usr/lib64/libcuda.so.1 /usr/local/cuda/lib64/stubs/libcuda.so 2>/dev/null | head -1", shell=True, capture_output=True, text=True).stdout.strip()
        if not found:
            found = subprocess.run("find / -name 'libcuda.so*' -not -path '*/kaggle/tmp/*' 2>/dev/null | head -1", shell=True, capture_output=True, text=True).stdout.strip()
        print("LIBCUDA", found or "NOT FOUND", flush=True)
        if found: os.symlink(found, stub + "/libcuda.so")
    env["LIBRARY_PATH"] = stub + ":" + env.get("LIBRARY_PATH", "")
    env["LD_LIBRARY_PATH"] = stub + ":" + env["LD_LIBRARY_PATH"]
    # Compiled flashinfer JIT + Triton caches built once on L4 by the attached sgl-jit-cache-l4 kernel; copying them in
    # (fresh mtimes, same /kaggle/tmp/sgl_site install path) skips the ~500 s first-server compile.
    fi = os.path.expanduser("~/.cache/flashinfer")
    if not os.path.isdir(env["TRITON_CACHE_DIR"]):
        # kernel outputs mount at /kaggle/input/<slug>/ or /kaggle/input/notebooks/<user>/<slug>/ depending on the session
        hits = glob.glob("/kaggle/input/*/sgl_cache") + glob.glob("/kaggle/input/notebooks/*/*/sgl_cache")
        if hits:
            shutil.copytree(hits[0] + "/triton", env["TRITON_CACHE_DIR"], dirs_exist_ok=True)
            shutil.copytree(hits[0] + "/flashinfer", fi, dirs_exist_ok=True)
            tvm = os.path.expanduser("~/.cache/tvm-ffi")   # sgl_kernel's JIT-compiled CUDA ops for arch 8.9 (the ~400 s of the first server start)
            if os.path.isdir(hits[0] + "/tvm-ffi"): shutil.copytree(hits[0] + "/tvm-ffi", tvm, dirs_exist_ok=True)
            print("CACHE_RESTORED from", hits[0], subprocess.run(f"du -sh {fi} {env['TRITON_CACHE_DIR']} {tvm} 2>/dev/null", shell=True, capture_output=True, text=True).stdout.replace(chr(10), " | "), flush=True)
        else:
            print("CACHE_MISSING; /kaggle/input:", os.listdir("/kaggle/input"), flush=True)
    return env


def start_server(gpu, port, max_lora_rank):
    env = sgl_env(); env["CUDA_VISIBLE_DEVICES"] = GPU_GROUPS[gpu]
    cmd = [sys.executable, "-m", "sglang.launch_server", "--model-path", GPTQ_MODEL, "--tokenizer-path", TOKENIZER,
           "--quantization", "compressed-tensors", "--dtype", "bfloat16", "--tp", "2", "--host", "127.0.0.1", "--port", str(port),
           "--context-length", str(MAX_SEQ + 1024), "--enable-lora", "--max-loras-per-batch", "2", "--max-lora-rank", str(max_lora_rank),
           "--lora-target-modules", *LORA_TARGETS, "--attention-backend", "flashinfer", "--kv-cache-dtype", "fp8_e4m3", "--sampling-backend", "pytorch",
           "--skip-server-warmup", "--cuda-graph-max-bs-decode", "48", "--schedule-policy", "lpm", "--schedule-conservativeness", "1.3", "--max-running-requests", "49",
           "--mamba-radix-cache-strategy", "extra_buffer_lazy", "--max-mamba-cache-size", "192", "--mamba-ssm-dtype", "bfloat16", "--disable-custom-all-reduce",
           "--chunked-prefill-size", "2048", "--cuda-graph-backend-prefill", "disabled", "--mem-fraction-static", "0.85", "--log-level", "info"]
    # 48 seats (192 Mamba slots, 49 running, decode graphs to bs 48) and mem fraction 0.85
    # (0.90 OOMed in sglang copy_mamba_state at 96 samples; 0.85 leaves room for the mamba copy-on-write buffers).
    cmd = filter_flags(cmd, env)
    os.makedirs(LOG_DIR, exist_ok=True)
    log = open(f"{LOG_DIR}/server_{gpu}.log", "w")
    return subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log


_HELP = {}
def filter_flags(cmd, env):
    """Drop flags this sglang version does not know (flag names moved between 0.5.x releases) instead of dying at startup."""
    if "text" not in _HELP:
        r = subprocess.run([sys.executable, "-m", "sglang.launch_server", "--help"], env=env, capture_output=True, text=True)
        _HELP["text"] = r.stdout + r.stderr
    out, i = cmd[:4], 4  # keep interpreter, -m, module, --model-path
    while i < len(cmd):
        tok = cmd[i]
        if tok.startswith("--"):
            j = i + 1
            while j < len(cmd) and not cmd[j].startswith("--"): j += 1
            if tok in _HELP["text"]: out.extend(cmd[i:j])
            else: print(f"FLAG_DROPPED {tok} {cmd[i+1:j]}", flush=True)
            i = j
        else:
            out.append(tok); i += 1
    return out


def wait_health(procs, ports, timeout=900):
    t0 = time.time(); up = set()
    while time.time() - t0 < timeout and len(up) < len(ports):
        for i, (p, port) in enumerate(zip(procs, ports)):
            if i in up: continue
            if p.poll() is not None:
                print("MEM", subprocess.run("free -g | sed -n 2p", shell=True, capture_output=True, text=True).stdout.strip(), flush=True)
                print(f"SERVER_DIED gpu{i} exit {p.returncode}; log tail:", flush=True)
                print(open(f"{LOG_DIR}/server_{i}.log").read()[-4000:], flush=True); raise SystemExit(2)
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=3).status_code == 200:
                    up.add(i); print(f"SERVER_UP gpu{i} port {port} after {time.time()-t0:.0f}s", flush=True)
            except requests.RequestException:
                pass
        time.sleep(3)
    if len(up) < len(ports):
        for i in range(len(ports)):
            if i not in up: print(f"SERVER_TIMEOUT gpu{i}; log tail:", open(f"{LOG_DIR}/server_{i}.log").read()[-3000:], flush=True)
        raise SystemExit(3)


def parse_grid(ids, formatter):
    ids = list(ids)
    if EOS_ID in ids: ids = ids[:ids.index(EOS_ID) + 1]
    return formatter.convert_tokens_to_array(ids)


def sample_puzzle(key, server, arc_test_set, formatter, tokenizer, limit, n_samples, views, end_time):
    t0 = time.time()
    r = requests.post(f"{server}/load_lora_adapter", json={"lora_name": key, "lora_path": f"{ADAPTER_DIR}/{key}"}, timeout=900)
    if r.status_code != 200:
        print(f"LORA_LOAD_FAIL {key} {r.status_code} {r.text[:200]}", flush=True); return
    puzzle_multi = arc_test_set.change_keys([key]).split_multi_replies()
    eval_ds = puzzle_multi.augment(n=2, seed=2).cut_to_len(formatter=formatter, name="input", max_len=MAX_SEQ - limit)
    by_test = collections.defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        by_test[subkey.split(".")[0]].append(subkey)
    for bk, subkeys in by_test.items():
        subkeys = subkeys[:views]
        if time.time() > end_time: print(f"TIME_UP before {bk}", flush=True); break
        tt = time.time(); gen_tokens = 0; grids = collections.Counter(); invalid = 0; nseq = 0
        def one(subkey):
            ids = tokenizer.encode(eval_ds.get(subkey, formatter)["input"])
            sp = {"temperature": 1.0, "top_p": 1.0, "top_k": -1, "max_new_tokens": limit, "stop_token_ids": [EOS_ID], "skip_special_tokens": False}
            # Copies are sent with n=1 (sglang 0.5.17 releases the LoRA usage counter once per child but acquires once per
            # parent, so n>1 wedges /unload_lora_adapter). Staged: one streaming request per view first; once it emits a
            # token the view's prompt KV is fully cached, then the other n_samples-1 copies are sent and hit the radix cache.
            # Sending all copies at once let LPM admit copies as extends of a half-prefilled prefix (1.56x prefill work).
            def send_rest():
                rr = requests.post(f"{server}/generate", json={"input_ids": [ids] * (n_samples - 1), "lora_path": [key] * (n_samples - 1), "sampling_params": sp}, timeout=1800)
                rr.raise_for_status(); ro = rr.json(); return ro if isinstance(ro, list) else [ro]
            acc = []; last = None; rest = None
            with ThreadPoolExecutor(max_workers=1) as rex:
                with requests.post(f"{server}/generate", json={"input_ids": ids, "lora_path": key, "sampling_params": sp, "stream": True}, stream=True, timeout=1800) as r:
                    r.raise_for_status()
                    for line in r.iter_lines():
                        if not line or not line.startswith(b"data:"): continue
                        body = line[5:].strip()
                        if body == b"[DONE]": break
                        o = json.loads(body); last = o; oid = o.get("output_ids") or []
                        if acc and len(oid) >= len(acc) and oid[:len(acc)] == acc: acc = list(oid)   # cumulative stream
                        else: acc.extend(oid)                                                        # incremental stream
                        if rest is None and acc and n_samples > 1: rest = rex.submit(send_rest)
                outs = [{"output_ids": acc, "meta_info": (last or {}).get("meta_info") or {"completion_tokens": len(acc)}}]
                if n_samples > 1:
                    if rest is None: rest = rex.submit(send_rest)   # first stream produced no token: send the copies anyway
                    outs += rest.result()
            return subkey, outs
        with ThreadPoolExecutor(max_workers=len(subkeys)) as ex:
            results = list(ex.map(one, subkeys))
        for subkey, outs in results:
            decoded = []
            for o in outs:
                nseq += 1; gen_tokens += (o.get("meta_info") or {}).get("completion_tokens", 0)
                ids = o.get("output_ids") or o.get("token_ids") or []
                grid = parse_grid(ids, formatter)
                if grid is None: invalid += 1; continue
                sol = puzzle_multi.invert_mod(grid, subkey, inv_perm=True)
                grids[tuple(map(tuple, sol.tolist()))] += 1
                decoded.append({"beam_score": 0.0, "score_aug": [0.0], "solution": sol})
            if decoded:
                with bz2.BZ2File(os.path.join(OUT_DIR, subkey), "w") as f: pickle.dump(decoded, f)
        el = time.time() - tt; top = grids.most_common(2)
        print(f"SAMPLE {bk} views {len(subkeys)} x {n_samples} = {nseq} in {el:.1f}s {gen_tokens/max(el,1e-6):.0f} tok/s uniq {len(grids)} invalid {invalid} top {[c for _, c in top]}", flush=True)
    try:
        requests.post(f"{server}/unload_lora_adapter", json={"lora_name": key}, timeout=120)
    except requests.RequestException as e:
        print(f"UNLOAD_WARN {key} {e}", flush=True)
    shutil.rmtree(f"{ADAPTER_DIR}/{key}", ignore_errors=True)
    print(f"PUZZLE_DONE {key} in {time.time()-t0:.1f}s", flush=True)


def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--end-time", type=float, required=True); ap.add_argument("--samples", type=int, default=8)
    ap.add_argument("--views", type=int, default=16); a = ap.parse_args()
    os.makedirs(OUT_DIR, exist_ok=True)
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/" + ("arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json")
    arc_test_set = ArcDataset.from_file(test_path)
    tokenizer = load_qwen2_fast_tokenizer(TOKENIZER); formatter = QwenFormatter(tokenizer=tokenizer); limit = formatter.max_new_tokens()
    keys = sorted(k for k in os.listdir(ADAPTER_DIR) if os.path.isfile(f"{ADAPTER_DIR}/{k}/adapter_model.safetensors") and k in arc_test_set.keys)
    rank = json.load(open(f"{ADAPTER_DIR}/{keys[0]}/adapter_config.json"))["r"] if keys else 32
    print(f"SAMPLING_START {len(keys)} puzzles, lora rank {rank}, {a.views} views x {a.samples} samples, limit {limit}, budget {(a.end_time-time.time())/3600:.2f} h", flush=True)
    ports = [30000 + i for i in range(len(GPU_GROUPS))]
    procs = []
    # all servers start together (host RAM is 188 GB).
    # With the restored kernel cache no server compiles; without it flashinfer's file locks make the servers share one build.
    print("MEM", subprocess.run("free -g | sed -n 2p", shell=True, capture_output=True, text=True).stdout.strip(), flush=True)
    for i, port in enumerate(ports):
        p, _ = start_server(i, port, rank); procs.append(p)
    wait_health(procs, ports, timeout=1200)
    print("MEM", subprocess.run("free -g | sed -n 2p", shell=True, capture_output=True, text=True).stdout.strip(), flush=True)
    q = queue.Queue()
    for k in keys: q.put(k)
    def worker(i):
        server = f"http://127.0.0.1:{ports[i]}"
        while time.time() < a.end_time - 60:
            try: key = q.get_nowait()
            except queue.Empty: return
            try:
                sample_puzzle(key, server, arc_test_set, formatter, tokenizer, limit, a.samples, a.views, a.end_time)
            except Exception as e:
                print(f"PUZZLE_ERROR {key} gpu{i}: {type(e).__name__}: {str(e)[:300]}", flush=True)
    # two driver threads per server: puzzle k+1 fills the lanes while puzzle k drains (S48 = 48 samples on a 24-lane server ran as
    # two waves with the batch at 16 lanes on average); the second adapter needs --max-loras-per-batch 2.
    threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(len(GPU_GROUPS)) for _ in range(2)]
    t0 = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    for src, dst in [(f"{LOG_DIR}/server_0.log", "/kaggle/working/server_0.log")] + [(j, "/kaggle/working/flashinfer_jit.log") for j in glob.glob(os.path.expanduser("~/.cache/flashinfer/**/flashinfer_jit.log"), recursive=True)]:
        if os.path.exists(src): shutil.copy(src, dst)
    print(f"SAMPLING_DONE {len(keys)} puzzles in {time.time()-t0:.0f}s; outputs {len(os.listdir(OUT_DIR))}", flush=True)
    for p in procs: p.terminate()
    time.sleep(5)
    for p in procs:
        if p.poll() is None: p.kill()


if __name__ == "__main__":
    main()


In [ ]:
sampling_start = time.time()
!python sgl_stage.py --end-time {global_end_time} --samples 3 --views 16 2>&1 | grep -vE "^\s*$" | tee /kaggle/working/sgl_stage.log | grep -E "SAMPLING_START|SERVER_|SAMPLE |PUZZLE_DONE|PUZZLE_ERROR|LORA_LOAD_FAIL|TIME_UP|SAMPLING_DONE|Traceback|Error"
print(f"*** SAMPLING STAGE: {time.time()-sampling_start:.0f}s")
!tail -5 /kaggle/tmp/sgl_logs/server_0.log


In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

# Time catch: a complete submission with [[0]] attempts is written first, so any task whose adapter or sampling did not
# finish before the deadline scores 0 instead of failing the run; the decoded results then overwrite what was produced.
submission = data.get_submission()
with open("submission.json", "w") as f:
    json.dump(submission, f)
try:
    if time.time() > global_end_time - 120: raise TimeoutError("time limit reached: all-[[0]] submission kept for every task")
    decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
    decoder.load_decoded_results("/kaggle/inference_outputs")
    submission = data.get_submission(decoder.run_selection_algo())
    with open("submission.json", "w") as f:
        json.dump(submission, f)
except Exception as e:
    print("*** DECODE FAILED, default submission kept:", repr(e), flush=True)
filled = sum(1 for k, v in submission.items() for t in v if t["attempt_1"] != [[0]] or t["attempt_2"] != [[0]])
print(f"*** SUBMISSION: {len(submission)} tasks, {sum(len(v) for v in submission.values())} test inputs, {filled} with predictions, elapsed {time.time()-global_start_time:.0f}s", flush=True)

if not rerun_mode:
    cv_score = data.validate_submission(submission)
    print(f"*** FINAL CV Pass@2: {cv_score:.6f}/{len(data.keys)} ({100*cv_score/len(data.keys):.4f}%)", flush=True)
    print(f"*** TOTAL ELAPSED: {time.time()-global_start_time:.2f}s", flush=True)
    try:
        decoder.benchmark_selection_algos()
    except Exception as e:
        print("benchmark_selection_algos skipped:", e)
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))


## Flash rule A
After the 27B submission is written, Flash-Next po352 (one model, two variations, symlinked) re-solves the 27B's least-sure 20% of test outputs in the time left: one FIFO queue of (task, attempt), 28 seats, a task is requeued until a verified Flash answer (check() passed) matches any 27B candidate or repeats 3 times. Final order per test output: matches -> Flash >= 2x -> 27B top-1 -> single verified Flash -> 27B top-2. Any failure keeps the 27B submission. Inputs: `flash-vllm-runtime-r1026b`, `flash-harness-fnt07cs72-r1027` (launcher + rule-A driver), `qwen3-8-flash-next-nvfp4-reap-po352` part1 + part2. Interactive commit: 8-puzzle smoke with the Flash cutoff at start + 90 min; hidden rerun: Flash cutoff 15 min before the 12 h deadline.

In [ ]:
%%writefile flash_ensemble.py
"""Flash rule A on top of the 27B: base candidate pools, task selection and the final merge. Stdlib + numpy only.

Pools: per test output ("task_i"), the 27B sampling's distinct grids with their vote counts, the submission's attempt_1/attempt_2
first, then the other grids by votes. Selection: the least-sure FRACTION of test outputs by top-1 votes (ties: smaller top-1/top-2
gap); tasks with more than 2 test inputs stay with the 27B. Merge per test output of a selected task, first two distinct of:
verified Flash answers that equal a 27B candidate (most votes first) -> verified Flash answers seen >= 2 times -> 27B top-1 ->
other verified Flash answers -> 27B top-2 -> rest. "Verified" = the attempt's check() passed (auto answer).
"""
import collections
import math


def _key(g):
    return tuple(tuple(int(c) for c in row) for row in g)


def norm_text(text):
    if text is None:
        return None
    rows = ["".join(ch for ch in r if ch.isdigit()) for r in str(text).strip().split("\n")]
    rows = [r for r in rows if r]
    return "\n".join(rows) or None


def text_of(g):
    return "\n".join("".join(str(int(c)) for c in row) for row in g)


def grid_of(text):
    return [[int(c) for c in row] for row in text.split("\n")]


def build_pools(decoder, submission):
    """{task_i: [[grid, votes], ...]} best first, from ArcDecoder.decoded_results (27B samples) and the written submission."""
    pools = {}
    for task, outs in submission.items():
        for i, att in enumerate(outs):
            key = f"{task}_{i}"
            votes = collections.Counter()
            for s in (decoder.decoded_results.get(key, {}) or {}).values():
                sol = s.get("solution") if isinstance(s, dict) else None
                if sol is not None:
                    votes[_key(sol.tolist() if hasattr(sol, "tolist") else sol)] += 1
            first = [_key(att[a]) for a in ("attempt_1", "attempt_2") if att.get(a) not in (None, [[0]]) and votes.get(_key(att[a]))]
            rest = sorted((g for g in votes if g not in first), key=lambda g: -votes[g])
            pools[key] = [[[list(r) for r in g], votes[g]] for g in dict.fromkeys(first + rest)]
    return pools


def select_tasks(pools, fraction, n_tests):
    """Least-sure tasks first; n_tests: {task: number of test inputs}."""
    def conf(k):
        v = [n for _, n in pools[k]]
        return (v[0] if v else -1, (v[0] - v[1]) if len(v) > 1 else 10 ** 9)
    outs = sorted(pools, key=conf)[:math.ceil(fraction * len(pools))]
    tasks = []
    for k in outs:
        t = k.rsplit("_", 1)[0]
        if n_tests[t] <= 2 and t not in tasks:
            tasks.append(t)
    return tasks


def merge_rule_a(submission, pools, answers, tasks):
    """answers: the launcher's answers.json {task: {test_idx: [{answer, auto, ...}]}}. Returns (new submission, changed outputs)."""
    new = {t: [dict(a) for a in v] for t, v in submission.items()}
    changed = 0
    for t in tasks:
        for i in range(len(submission[t])):
            pool = [text_of(g) for g, _ in pools.get(f"{t}_{i}", [])]
            rank = {g: r for r, g in enumerate(pool)}
            votes = collections.Counter(g for g in (norm_text(x.get("answer")) for x in answers.get(t, {}).get(str(i), []) if x.get("auto"))
                                        if g is not None)
            matched = sorted((g for g in votes if g in rank), key=lambda g: (-votes[g], rank[g]))
            multi = sorted((g for g in votes if votes[g] >= 2 and g not in matched), key=lambda g: -votes[g])
            single = sorted((g for g in votes if g not in matched and g not in multi), key=lambda g: -votes[g])
            final = []
            for g in matched + multi + pool[:1] + single + pool[1:]:
                if g not in final:
                    final.append(g)
                if len(final) == 2:
                    break
            if not final:
                continue
            cur = new[t][i]
            a1, a2 = grid_of(final[0]), grid_of(final[1]) if len(final) > 1 else cur.get("attempt_2", [[0]])
            if [a1, a2] != [cur.get("attempt_1"), cur.get("attempt_2")]:
                changed += 1
            new[t][i] = {"attempt_1": a1, "attempt_2": a2}
    return new, changed

In [ ]:
# Flash rule A: Flash-Next po352 re-solves the 27B's least-sure tasks in the time left, then merges into submission.json.
# Uses `submission` / `decoder` from the cell above; any failure keeps the 27B submission already written.
import glob, json, os, subprocess, sys, time
from flash_ensemble import build_pools, select_tasks, merge_rule_a

FLASH_FRACTION = 0.20                              # least-sure share of test outputs (tasks with > 2 test inputs stay 27B)
FLASH_CUTOFF = global_end_time - 15 * 60 if rerun_mode else global_start_time + 90 * 60   # stop Flash here; interactive commit = 8-task smoke
FLASH_SETTING, FLASH_MAXLEN, FLASH_SEATS = "ov_l4_fnt07cs72_a", 98304, 28
FLASH_MIN_LEFT = 30 * 60                           # runtime unpack (~2 min) + server load (~9 min on Kaggle) + useful attempts
FLASH_OUT = "/kaggle/working/flash"

def one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"input not found: {pattern}"
    return hits[0]

flash_t0 = time.time()
try:
    if time.time() > FLASH_CUTOFF - FLASH_MIN_LEFT:
        raise TimeoutError(f"only {(FLASH_CUTOFF - time.time())/60:.0f} min left before the Flash cutoff")
    os.makedirs(FLASH_OUT, exist_ok=True)
    pools = build_pools(decoder, submission)
    n_tests = {t: len(v) for t, v in submission.items()}
    flash_tasks = select_tasks(pools, FLASH_FRACTION, n_tests)
    json.dump(pools, open(f"{FLASH_OUT}/pools.json", "w"))
    json.dump({"keys": flash_tasks}, open(f"{FLASH_OUT}/keys.json", "w"))
    print(f"FLASH: {len(flash_tasks)} tasks selected (least sure first) from {len(pools)} test outputs; "
          f"cutoff in {(FLASH_CUTOFF - time.time())/60:.0f} min", flush=True)
    runtime = one("/kaggle/input/**/flash-runtime-r1026.part-00")
    harness = os.path.dirname(one("/kaggle/input/**/flash-harness*/flash_kaggle_run.py"))
    p1 = os.path.dirname(one("/kaggle/input/models/**/nvfp4-po352-kaggle-4gb-part1/**/config.json"))
    p2 = os.path.dirname(one("/kaggle/input/models/**/nvfp4-po352-kaggle-4gb-part2/**/model-00030-of-00030.safetensors"))
    model = "/kaggle/tmp/po352_nvfp4"
    os.makedirs(model, exist_ok=True)
    for d in (p1, p2):
        for f in os.listdir(d):
            if not os.path.exists(os.path.join(model, f)):
                os.symlink(os.path.join(d, f), os.path.join(model, f))
    comp = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
    challenges = f"{comp}/arc-agi_test_challenges.json" if rerun_mode else f"{comp}/arc-agi_evaluation_challenges.json"
    cmd = [sys.executable, os.path.join(harness, "flash_kaggle_run.py"), "--runtime", runtime, "--harness", harness, "--model", model,
           "--challenges", challenges, "--keys", f"{FLASH_OUT}/keys.json", "--pools", f"{FLASH_OUT}/pools.json", "--out", FLASH_OUT,
           "--setting", FLASH_SETTING, "--maxlen", str(FLASH_MAXLEN), "--seats", str(FLASH_SEATS), "--offload-gib", "32",
           "--deadline", str(FLASH_CUTOFF - 120), "--progress-every", "300"]
    if os.environ.get("NCCL_P2P_DISABLE") != "1":
        cmd.append("--p2p")                        # Kaggle 4xL4 has working P2P (hosts without it need NCCL_P2P_DISABLE=1)
    try:   # the 27B's sglang servers must be gone before vLLM takes 96% of each GPU
        subprocess.run(["pkill", "-9", "-f", r"sglang\.launch_server|sglang::"], check=False)
        for _ in range(60):
            used = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                                   capture_output=True, text=True).stdout.split()]
            if used and max(used) < 1024:
                break
            time.sleep(1)
        print(f"FLASH: GPU MiB used before launch {used}", flush=True)
    except Exception as e:
        print("FLASH: GPU check skipped:", repr(e), flush=True)
    try:
        subprocess.run(cmd, timeout=max(60, FLASH_CUTOFF - time.time() + 60))   # hard stop 1 min after the cutoff
    except subprocess.TimeoutExpired:   # launcher overran; merge the attempts that finished (half-written last line skipped)
        res = {}
        for line in open(f"{FLASH_OUT}/shard0.jsonl"):
            try:
                r = json.loads(line)
            except ValueError:
                continue
            tid, idx = r["puzzle_id"].split("#")
            res.setdefault(tid, {}).setdefault(idx, []).append({"answer": r.get("answer"), "auto": bool(r.get("auto_answer"))})
        json.dump(res, open(f"{FLASH_OUT}/answers.json", "w"))
        print("FLASH: hard timeout; answers.json rebuilt from shard0.jsonl", flush=True)
    answers = json.load(open(f"{FLASH_OUT}/answers.json"))
    merged, changed = merge_rule_a(submission, pools, answers, flash_tasks)
    with open("submission.json.tmp", "w") as f:
        json.dump(merged, f)
    os.replace("submission.json.tmp", "submission.json")
    n_att = sum(len(v) for t in answers.values() for v in t.values())
    print(f"*** FLASH: {len(answers)} tasks answered, {n_att} attempt records, {changed} test outputs changed, "
          f"{(time.time() - flash_t0)/60:.1f} min", flush=True)
    if not rerun_mode:
        print(f"*** 27B only CV Pass@2: {data.validate_submission(submission):.6f}/{len(data.keys)}", flush=True)
        print(f"*** 27B + Flash CV Pass@2: {data.validate_submission(merged):.6f}/{len(data.keys)}", flush=True)
    submission = merged
except Exception as e:
    print("*** FLASH SKIPPED or FAILED, 27B submission kept:", repr(e), flush=True)
print(f"*** TOTAL ELAPSED (with Flash): {time.time() - global_start_time:.2f}s", flush=True)